In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

In [2]:
df = pd.read_csv('crop_data.csv')
df.head()

,N,P,K,temperature,humidity,pH,rainfall,label
0,29,13,11,13.79,94.30,6.79,116.01,orange
1,25,29,27,27.96,84.74,6.23,176.42,coconut
2,28,76,21,34.13,63.92,7.41,73.30,blackgram
3,112,8,54,31.98,93.25,6.45,21.14,muskmelon
4,6,12,6,12.20,91.23,6.86,115.26,orange


In [3]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2200 entries, 0 to 2199
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   N            2200 non-null   int64  
 1   P            2200 non-null   int64  
 2   K            2200 non-null   int64  
 3   temperature  2200 non-null   float64
 4   humidity     2200 non-null   float64
 5   pH           2200 non-null   float64
 6   rainfall     2200 non-null   float64
 7   label        2200 non-null   object 
dtypes: float64(4), int64(3), object(1)
memory usage: 137.6+ KB


,N,P,K,temperature,humidity,pH,rainfall
count,2200.000000,2200.000000,2200.000000,2200.000000,2200.000000,2200.000000,2200.000000
mean,50.672727,51.853182,47.435455,26.190395,71.406732,6.421573,103.620186
std,36.566872,32.752059,50.596267,5.522292,22.488904,0.727238,54.667720
min,0.000000,0.000000,5.000000,8.440000,10.160000,3.630000,20.150000
25%,21.000000,28.000000,20.000000,22.670000,60.890000,5.940000,64.970000
50%,37.000000,48.000000,31.000000,25.770000,79.990000,6.400000,95.890000
75%,85.000000,67.000000,47.000000,29.682500,89.755000,6.910000,124.435000
max,139.000000,144.000000,204.000000,43.940000,99.880000,9.490000,297.800000


In [4]:
X = df.drop('label', axis=1)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [5]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=50)
rf_model.fit(X_train_scaled, y_train)

y_predictions = rf_model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_predictions)
print(f"Model Accuracy: {accuracy * 100:.2f} %\n")
print(classification_report(y_test, y_predictions))

Model Accuracy: 99.09 %

              precision    recall  f1-score   support

       apple       1.00      1.00      1.00        20
      banana       1.00      1.00      1.00        15
   blackgram       1.00      1.00      1.00        28
    chickpea       1.00      1.00      1.00        18
     coconut       1.00      1.00      1.00        15
      coffee       1.00      1.00      1.00        25
      cotton       1.00      1.00      1.00        30
      grapes       1.00      1.00      1.00        21
        jute       1.00      0.95      0.98        21
 kidneybeans       1.00      1.00      1.00        13
      lentil       0.92      1.00      0.96        22
       maize       1.00      1.00      1.00        19
       mango       1.00      1.00      1.00        24
   mothbeans       1.00      0.91      0.95        23
    mungbean       1.00      1.00      1.00        27
   muskmelon       1.00      1.00      1.00        16
      orange       1.00      1.00      1.00        12
  

In [6]:
def get_top_5_crops(soil_data_scaled, model):
    probabilities = model.predict_proba(soil_data_scaled)[0]
    crop_names = model.classes_
    
    results = pd.DataFrame({
        'Crop': crop_names,
        'Probability (%)': probabilities * 100
    })
    
    top_5 = results.sort_values(by='Probability (%)', ascending=False).head(5)
    top_5['Probability (%)'] = top_5['Probability (%)'].round(1).astype(str) + " %"
    return top_5

In [7]:
new_farm_perfect = pd.DataFrame(
    [[104, 18, 50, 23.6, 60.3, 6.7, 140.9]], 
    columns=X.columns
)

new_farm_scaled = scaler.transform(new_farm_perfect)
top_crops_table = get_top_5_crops(new_farm_scaled, rf_model)
print(top_crops_table.to_string(index=False))

       Crop Probability (%)
     coffee          63.0 %
       jute          14.0 %
 watermelon          14.0 %
     banana           5.0 %
pomegranate           2.0 %


In [8]:
crop_characteristics = df.groupby('label').mean().round(2)
crop_characteristics

,N,P,K,temperature,humidity,pH,rainfall
label,,,,,,,
apple,20.32,131.73,199.18,22.47,92.38,5.96,113.48
banana,99.76,81.27,49.48,30.09,79.92,6.19,104.54
blackgram,39.73,67.31,19.67,29.82,65.25,7.13,70.51
chickpea,38.93,67.05,79.60,19.25,15.22,7.06,75.17
coconut,20.17,18.60,29.38,27.12,88.92,5.73,181.01
coffee,100.78,27.96,29.19,25.48,60.94,6.84,156.41
cotton,118.17,44.92,19.62,23.76,80.22,6.43,79.27
grapes,20.69,132.46,199.51,23.31,82.33,6.00,70.12
jute,79.53,44.15,39.74,24.86,80.13,6.79,174.43


In [9]:
try:
    user_N = float(input("Enter Nitrogen (N) content: "))
    user_P = float(input("Enter Phosphorus (P) content: "))
    user_K = float(input("Enter Potassium (K) content: "))
    user_temp = float(input("Enter Temperature (C): "))
    user_hum = float(input("Enter Humidity (%): "))
    user_pH = float(input("Enter pH level: "))
    user_rain = float(input("Enter Rainfall (mm): "))

    user_soil_data = pd.DataFrame(
        [[user_N, user_P, user_K, user_temp, user_hum, user_pH, user_rain]],
        columns=['N', 'P', 'K', 'temperature', 'humidity', 'pH', 'rainfall']
    )

    user_scaled = scaler.transform(user_soil_data)
    
    print("\nResults:")
    top_recommendations = get_top_5_crops(user_scaled, rf_model)
    print(top_recommendations.to_string(index=False))

except ValueError:
    print("Error: Please enter valid numbers only.")



Results:
      Crop Probability (%)
     mango          29.0 %
pigeonpeas          23.0 %
  chickpea          21.0 %
    papaya          12.0 %
    banana           5.0 %
